In [1]:
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.chains import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory, RunnablePassthrough
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

True

Load the clean vector store and build the retriever

In [2]:
# absolute path
persist_dir = r'C:\Users\USER\rag_course\chroma_db_clean'    

# Embedding model
embeddings = OpenAIEmbeddings()

# Load existing vector store
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings)

# Retriever:return top 4 similar chunks
#retriever = vectorstore.as_retriever(search_kwargs={'k': 8})

retriever = vectorstore.as_retriever(
    search_kwargs={
        'k': 8,
        'filter': {'file_name': 'nigeria_health_diseases_and_prevention.pdf'}
    }
)

print('Retriever ready')
print(f'Vectors in store: {vectorstore._collection.count()}')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Retriever ready
Vectors in store: 80


Create the LLM and the two prompts

In [3]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You rewrite a user\'s latest question into a standalone question '
     'using the chat history.\n'
     '\n'
     'Rules:\n'
     '1. If the latest question refers to something from the history '
     '(like "the first one", "that", "it", "the second option"), REPLACE '
     'that reference with the actual item from the history.\n'
     '2. Do NOT answer the question. Only rewrite it.\n'
     '3. If the question is already standalone, return it unchanged.\n'
     '\n'
     'Examples:\n'
     '- History mentions "Cassava Mosaic Disease" first, then "Maize Smut".\n'
     '  User: "How do I control the first one?"\n'
     '  Rewritten: "How do I control Cassava Mosaic Disease?"\n'
     '\n'
     '- History lists "malaria" first, then "respiratory infection".\n'
     '  User: "Tell me more about the first one."\n'
     '  Rewritten: "Tell me more about malaria."'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. '
               'Answer the question using only the provided context. '
               'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}'),
])

print('✅ Improved prompts ready')

✅ Improved prompts ready


Build the history‑aware RAG chain

In [4]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def rewrite_or_pass(x):
    chat_history = x.get('chat_history') or []
    question = x['input']
    
    if not chat_history:
        return question
    
    rewritten = (rewrite_prompt | llm | StrOutputParser()).invoke({
        'chat_history': chat_history,
        'input': question,
    })
    return rewritten

# Join retrieved documents into a single context block
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# Full RAG chain
rag_chain = (
    {
        'context': RunnableLambda(rewrite_or_pass) | retriever | format_docs,
        'input': RunnableLambda(lambda x: x['input']),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

print('RAG chain ready')

RAG chain ready


Set up in‑memory chat history

In [5]:
# from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

store = {}

def get_history(session_id):
    
    # If session doesn't exist/new...
    if session_id not in store:
        
        # create a new history
        store[session_id] = []
        
    # Return the history for this session
    return store[session_id]

def ask(question, session_id):
    
    # Fetch all previous messages in this session
    history = get_history(session_id)
    
    # 1. Rewrite if there is history, otherwise pass question through
    if history:
        rewritten = (rewrite_prompt | llm | StrOutputParser()).invoke({
            'chat_history': history,   # Feed it the chat history
            'input': question     # current question.
        })
        
    # If no history    
    else:
        rewritten = question
        
    # 2. Retrieve documents with the rewritten question
    docs = retriever.invoke(rewritten)

    # 3. Join the list docs into one context block
    context = '\n\n'.join(doc.page_content for doc in docs)
    
    # 4. Generate the answer
    answer = (qa_prompt | llm | StrOutputParser()).invoke({
        'context': context,
        'input': rewritten,
    })
    
    # 5. Save this turn into history
    history.append(HumanMessage(content=question))   # Store user message
    history.append(AIMessage(content=answer))        # Store assistant message
    
    return answer
    
    

print('Manual conversational helper ready')

Manual conversational helper ready


Wrap the RAG chain with automatic history management

In [6]:
# Turn 1
session_id = 'test-1'
q1 = 'What are the top causes of death in Nigeria?'
a1 = ask(q1, session_id)
print('👤 User:', q1)
print('🤖 Assistant:', a1)
print('-' * 70)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


👤 User: What are the top causes of death in Nigeria?
🤖 Assistant: The top 10 causes of death in Nigeria are as follows:

- Malaria (20%)
- Lower Respiratory Infection (19%)
- HIV/AIDS (9%)
- Diarrheal Diseases (5%)
- Road Injuries (5%)
----------------------------------------------------------------------


Follow‑up question using history

In [7]:
session_id = 'test-1'                                     

# Turn 2 — vague follow-up
q2 = 'What about the first one you mentioned?'

# Reuse the ask() helper
a2 = ask(q2, session_id)                                          

print('👤 User:', q2)
print('🤖 Assistant:', a2[:400])
print('-' * 70)

👤 User: What about the first one you mentioned?
🤖 Assistant: Malaria is one of the top causes of death in Nigeria, accounting for 20% of deaths. The current national health policy includes concise statements on health programs related to malaria, among other health issues. However, the overall health status improvement in Nigeria, including the management of malaria, has been insignificant.
----------------------------------------------------------------------


In [8]:
session_id = 'test-1'                                     

# Turn 3 — vague follow-up
q3 = 'What about the last one you mentioned?'

# Reuse the ask() helper
a3 = ask(q3, session_id)                                          

print('👤 User:', q3)
print('🤖 Assistant:', a3[:400])
print('-' * 70)

👤 User: What about the last one you mentioned?
🤖 Assistant: The statement about road injuries accounting for 5% of deaths in Nigeria is accurate, as mentioned in the context. However, the specific details regarding ongoing efforts to improve road safety and the challenges in addressing this issue are not provided in the context. Therefore, I cannot elaborate further on those efforts or challenges.
----------------------------------------------------------------------
